# PA model: false alarms -> PA history -> results (live demo)
Run top to bottom from `Global_prd_rules/`. Each cell is short on purpose: change a value and rerun.
**No patient IDs, visit IDs or dates are shown**: patients appear as "Patient 1", days as "day 0, day 7, ...".

In [ ]:
# 0. Setup
import numpy as np, pandas as pd, pyarrow.fs as pafs
from build_pa_history import family          # CPT -> family (THERAPY, INFUSION, EM, IMAG, ...)
pd.set_option("display.width", 200)
fs = pafs.S3FileSystem()
V11 = "cli-ml-data-prod/datalake/ai-prior-auth/09282026/std_data_v11_hist_draft/ai_prior_auth_09282026_v3_2026_std.parquet"
CORE, HIST = "exp_feat_core_s42", "exp_feat_hist_s42"

## Part 1 - Where are the false alarms?

In [ ]:
# 1.1 Load the core model's test scores, one row per service (visit + CPT)
def services(run):
    t = pd.read_parquet(f"{run}/test_preds.parquet")
    p = [c for c in t.columns if c.startswith("p_")][0]
    t = t[~t.FINAL_CPT.astype(str).str.fullmatch(r"\d{4}F")]                  # quality codes out
    return t.groupby(["CSN", "FINAL_CPT"]).agg(y=("y", "max"), score=(p, "max"),
                                                payer=("PAYOR_CLEAN", "first")).reset_index()
S = services(CORE)
S["family"] = S.FINAL_CPT.map(family)
len(S), S.y.sum(), round(S.y.mean(), 4)                                         # services, PA services, PA rate

In [ ]:
# 1.2 Cut-off that catches 85% of PA, and the four outcomes
cut = S.score[S.y == 1].quantile(0.15)
S["flag"] = S.score >= cut
S["outcome"] = np.select([S.flag & (S.y == 1), S.flag & (S.y == 0), ~S.flag & (S.y == 1)], ["TP", "FP", "FN"], "TN")
print(f"cut-off {cut:.3f}"); S.outcome.value_counts()

In [ ]:
# 1.3 Precision = TP / (TP + FP)
n = S.outcome.value_counts()
print(f"precision {n.TP / (n.TP + n.FP):.1%} | recall {n.TP / (n.TP + n.FN):.1%}")

In [ ]:
# 1.4 False alarms by CPT family
S[S.outcome == "FP"].family.value_counts().head(8)

In [ ]:
# 1.5 Overall, PA services score much higher than non-PA (the model works in general)
fam = "THERAPY"                                                                  # try INFUSION, HCPCS_J, IMAG
T = S[S.family == fam]
T.groupby("y").score.quantile([0.1, 0.5, 0.9]).unstack()

In [ ]:
# 1.6 But the flagged ones still include many non-PA services
T[T.flag].y.value_counts()                                                       # 1 = PA caught, 0 = false alarm

In [ ]:
# 1.7 Combinations that are mostly PA: the model scores everything in them high
top = T.groupby(["FINAL_CPT", "payer"]).y.agg(services="size", pa_rate="mean")
top[top.services > 300].sort_values("pa_rate", ascending=False).head(8)

In [ ]:
# 1.8 Inside ONE combination, PA and non-PA get the SAME score -> the non-PA ones become false alarms
cpt, payer = top[top.services > 300].sort_values("pa_rate", ascending=False).index[0]   # or type e.g. ("97140", "ANTHEM BCBS MEDICAID")
T[(T.FINAL_CPT == cpt) & (T.payer == payer)].groupby("y").score.describe()[["count", "mean", "50%"]]

## Part 2 - What is missing: where the patient is in their treatment

In [ ]:
# 2.1 Load the test months from the history data (lines)
cols = ["PAT_ID", "CSN", "FINAL_CPT", "SERVICEDATE", "LABEL_V6_PAX", "PAH_CPT_90D", "PAH_FAM_90D", "PAH_ANY_90D", "PAH_FAM_365D"]
A = pd.read_parquet(V11, columns=cols, filesystem=fs)                           # whole 2026 file (for 2.3)
A["date"] = pd.to_datetime(A.SERVICEDATE, errors="coerce")
A["pa"] = A.LABEL_V6_PAX.astype(str).str.strip().str.upper().eq("TRUE") & ~A.FINAL_CPT.astype(str).str.fullmatch(r"\d{4}F")
L = A[(A.date >= "2026-05-01") & (A.date < "2026-08-01") & ~A.FINAL_CPT.astype(str).str.fullmatch(r"\d{4}F")].copy()
L["family"] = L.FINAL_CPT.map(family)
len(L)

In [ ]:
# 2.2 One patient's therapy visits (anonymised): every visit of the course is PA
cpt = "97110"                                                                    # try 96365, J9271
d = L[L.FINAL_CPT == cpt].drop_duplicates(["PAT_ID", "date"])
pat = d[d.pa].PAT_ID.value_counts().index[0]                                     # patient with most PA visits
one = d[d.PAT_ID == pat].sort_values("date")
pd.DataFrame({"patient": "Patient 1", "day": (one.date - one.date.min()).dt.days,
              "PA": one.pa.values, "PA_days_same_CPT_last_90d": one.PAH_CPT_90D.values}).reset_index(drop=True).head(12)

In [ ]:
# 2.3 How the history count is made, by hand: count this patient's PA days 1-90 days BEFORE each visit
pa_days = A[(A.PAT_ID == pat) & (A.FINAL_CPT == cpt) & A.pa].date.drop_duplicates()   # incl. Feb-Apr, before the test months
[int(((x - pa_days).dt.days.between(1, 90)).sum()) for x in one.date][:12]       # should match the last column above

## Part 3 - Does history separate PA from non-PA?

In [ ]:
# 3.1 PA rate with vs without same-family history (lines, test months)
L["has_hist"] = L.PAH_FAM_90D > 0
t = L.groupby(["family", "has_hist"]).pa.mean().unstack().rename(columns={False: "no_history", True: "history"})
t.loc[["THERAPY", "INFUSION", "HCPCS_J", "IMAG", "EM"]].style.format("{:.1%}")

In [ ]:
# 3.2 How many lines have history (it is rare overall, but concentrated where PA is)
L.has_hist.mean(), L[L.pa].has_hist.mean()                                       # all lines vs PA lines

## Part 4 - Results

In [ ]:
# 4.1 False alarms at 85% recall, same test services, agreed labels
r = pd.read_csv("feats_confusion.csv")
keep = ["l6pax", "core_s42", "core_s7", "hist_s42", "hist_s7", "hist_lag30_s42"]
r = r[(r["answer key"] == "agreed") & (r["target recall"] == "85%") & r.model.isin(keep)]
r["precision"] = r.TP / (r.TP + r.FP)
r.set_index("model").loc[[m for m in keep if m in set(r.model)], ["TP", "FP", "FN", "precision"]]

In [ ]:
# 4.2 Where the false alarms went: core vs core + history, by family
f = pd.read_csv("full_review_hist/2_by_family.csv", index_col=0)
f[["FP_base", "FP_model", "FP_change"]].rename(columns={"FP_base": "FP_core", "FP_model": "FP_core_hist"}).head(8)

In [ ]:
# 4.3 Same comparison directly from the two runs (no review file needed)
H = services(HIST); H["family"] = H.FINAL_CPT.map(family)
cut_h = H.score[H.y == 1].quantile(0.15)
fp = lambda D, c: D[(D.score >= c) & (D.y == 0)].family.value_counts()
pd.DataFrame({"core": fp(S, cut), "core+history": fp(H, cut_h)}).fillna(0).astype(int).sort_values("core", ascending=False).head(8)

In [ ]:
# 4.4 One example: the same CPT + payer, core vs history model, service by service
k = ["CSN", "FINAL_CPT"]
fix = lambda D: D.assign(CSN=D.CSN.astype(str).str.replace(r"\.0$", "", regex=True))
hist = fix(L).groupby(k)[["PAH_CPT_90D", "PAH_FAM_90D"]].max().reset_index()
X = (fix(S)[k + ["y", "score", "payer", "family"]]
     .merge(fix(H)[k + ["score"]].rename(columns={"score": "score_hist"}), on=k)
     .merge(hist, on=k, how="left").fillna({"PAH_CPT_90D": 0, "PAH_FAM_90D": 0}))
X["core"] = np.where(X.score >= cut, np.where(X.y == 1, "TP", "FP"), np.where(X.y == 1, "FN", "TN"))
X["with_history"] = np.where(X.score_hist >= cut_h, np.where(X.y == 1, "TP", "FP"), np.where(X.y == 1, "FN", "TN"))
fixed = X[(X.core == "FP") & (X.with_history == "TN") & (X.PAH_FAM_90D == 0)]
cpt, payer = fixed.groupby(["FINAL_CPT", "payer"]).size().idxmax()       # combination where history removed most false alarms
ex = X[(X.FINAL_CPT == cpt) & (X.payer == payer)]
print(cpt, payer, "| services:", len(ex)); pd.crosstab(ex.core, ex.with_history, margins=True)

In [ ]:
# 4.5 Side by side: false alarms that history removed vs PA it still catches (same CPT + payer)
cols = ["FINAL_CPT", "payer", "PAH_FAM_90D", "y", "score", "core", "score_hist", "with_history"]
pd.concat([ex[(ex.core == "FP") & (ex.with_history == "TN") & (ex.PAH_FAM_90D == 0)].head(4),
           ex[(ex.core == "TP") & (ex.with_history == "TP") & (ex.PAH_FAM_90D > 0)].head(4)])[cols].round(3).reset_index(drop=True)

## Part 5 - The history columns in the data, and how often patients come back
Only counts and shares are shown: no patient IDs, visit IDs or exact dates.

In [ ]:
# 5.1 Sample rows: what the 4 new columns look like (test months)
show = ["FINAL_CPT", "family", "pa", "PAH_CPT_90D", "PAH_FAM_90D", "PAH_ANY_90D", "PAH_FAM_365D"]
pd.concat([L[L.PAH_FAM_90D > 0].sample(5, random_state=1), L[L.PAH_FAM_90D == 0].sample(5, random_state=1)])[show].reset_index(drop=True)

In [ ]:
# 5.2 One family only, e.g. therapy lines with history
L[(L.family == "THERAPY") & (L.PAH_FAM_90D > 0)][show].sample(8, random_state=2).reset_index(drop=True)

In [ ]:
# 5.3 Share of lines with history > 0, and the spread of values (test months)
print((L[["PAH_CPT_90D", "PAH_FAM_90D", "PAH_ANY_90D", "PAH_FAM_365D"]] > 0).mean().round(4))
L.PAH_FAM_90D.clip(upper=10).value_counts().sort_index()                       # 10 = 10 or more days

In [ ]:
# 5.4 Load all years (light columns) and mark the period of each line
YEARS = [2022, 2023, 2024, 2025, 2026]
V11_Y = V11.replace("_2026_", "_{year}_")
Y = pd.concat([pd.read_parquet(V11_Y.format(year=y), columns=["PAT_ID", "FINAL_CPT", "SERVICEDATE", "LABEL_V6_PAX", "PAH_FAM_90D"],
                               filesystem=fs) for y in YEARS], ignore_index=True)
Y["date"] = pd.to_datetime(Y.SERVICEDATE, errors="coerce")
Y = Y[~Y.FINAL_CPT.astype(str).str.fullmatch(r"\d{4}F")]
Y["pa"] = Y.LABEL_V6_PAX.astype(str).str.strip().str.upper().eq("TRUE")
Y["period"] = np.select([Y.date < "2026-05-01", Y.date < "2026-08-01"], ["train (to Apr 2026)", "test (May-Jul 2026)"], "after test")
Y.period.value_counts()

In [ ]:
# 5.5 How often patients come back - all visits (a visit day = patient + date)
def visits(D):
    v = D.drop_duplicates(["PAT_ID", "date"]).groupby("PAT_ID").size()
    return pd.Series({"patients": len(v), "visit_days": int(v.sum()), "median_days_per_patient": v.median(),
                      "share_1_day": (v == 1).mean(), "share_2_5": v.between(2, 5).mean(), "share_6_plus": (v >= 6).mean()})
pd.DataFrame({"overall": visits(Y), **{p: visits(Y[Y.period == p]) for p in ["train (to Apr 2026)", "test (May-Jul 2026)"]}}).round(3)

In [ ]:
# 5.6 Same, PA only: how many PA days per patient
P = Y[Y.pa]
pd.DataFrame({"overall": visits(P), **{p: visits(P[P.period == p]) for p in ["train (to Apr 2026)", "test (May-Jul 2026)"]}}).round(3)

In [ ]:
# 5.7 Share of PA lines that are part of a running course (same-family PA in the previous 90 days)
Y["family"] = Y.FINAL_CPT.map(family)
r = Y[Y.pa].assign(repeat=lambda d: d.PAH_FAM_90D > 0).groupby("period").repeat.mean()
print(r.round(3))
Y[Y.pa & (Y.period == "test (May-Jul 2026)")].assign(repeat=lambda d: d.PAH_FAM_90D > 0) \
  .groupby("family").repeat.agg(PA_lines="size", share_repeat="mean").sort_values("PA_lines", ascending=False).head(8).round(3)